<a href="https://colab.research.google.com/github/nothing248/notebooks/blob/main/notebooks/video_subtitle/remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title colab 挂载目录
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
# @title 定义路径
pro_path = /content/drive/MyDrive/projects
tmp_path = /content

In [ ]:
# @title 查看cuda版本
!nvcc -V
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
Mon Jul 20 06:04:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8       

In [ ]:
# @title 克隆仓库
!git clone --depth 1 https://github.com/YaoFANGUK/video-subtitle-remover {pro_path}/video-subtitle-remover

Cloning into '/content/drive/MyDrive/projects'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 230 (delta 14), reused 162 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 754.55 MiB | 9.10 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Updating files: 100% (184/184), done.


In [ ]:
# @title 安装uv
import os

%cd {tmp_path}

!pip install uv
os.environ["UV_VENV_CLEAR"] = "1"
!uv venv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 89.9 MB/s eta 0:00:00
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [ ]:
# @title 安装环境
!uv pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ --python .venv/bin/python
!uv pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118 --python .venv/bin/python
!uv pip install onnxruntime-gpu==1.22.0 --python .venv/bin/python
!uv pip install nvidia-cudnn-cu11==8.6.0.163 --python .venv/bin/python
!uv pip install -r /content/drive/MyDrive/projects/video-subtitle-remover/requirements.txt --python .venv/bin/python
!uv pip install pip setuptools --python .venv/bin/python
!uv run paddlex --install hpi-gpu
!uv pip install matplotlib-inline --python .venv/bin/python

Resolved 26 packages in 1.50s
Uninstalled 2 packages in 1ms
Installed 2 packages in 2ms
 - nvidia-cudnn-cu11==8.6.0.163
 + nvidia-cudnn-cu11==8.9.6.50
 - nvidia-nccl-cu11==2.21.5
 + nvidia-nccl-cu11==2.19.3
Resolved 25 packages in 312ms
Uninstalled 2 packages in 1ms
Installed 2 packages in 2ms
 - nvidia-cudnn-cu11==8.9.6.50
 + nvidia-cudnn-cu11==9.1.0.70
 - nvidia-nccl-cu11==2.19.3
 + nvidia-nccl-cu11==2.21.5
Checked 1 package in 2ms
Resolved 2 packages in 2ms
Uninstalled 1 package in 0.85ms
Installed 1 package in 1ms
 - nvidia-cudnn-cu11==9.1.0.70
 + nvidia-cudnn-cu11==8.6.0.163
Checked 15 packages in 12ms
Checked 2 packages in 2ms
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
/content/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and in

In [ ]:
# @title 查看帮助
%cd {path}/video-subtitle-remover
!/content/.venv/bin/python ./backend/main.py --help


📢 Tips: QFluentWidgets Pro is now released. Click https://qfluentwidgets.com/pages/pro to learn more about it.

usage: main.py [-h] --input INPUT [--output OUTPUT]
               [--subtitle-area-coords YMIN YMAX XMIN XMAX]
               [--inpaint-mode {sttn-auto,sttn-det,lama,propainter,opencv}]

Video Subtitle Remover Command Line Tool

options:
  -h, --help            show this help message and exit
  --input INPUT, -i INPUT
                        Input video file path
  --output OUTPUT, -o OUTPUT
                        Output video file path (optional)
  --subtitle-area-coords YMIN YMAX XMIN XMAX, -c YMIN YMAX XMIN XMAX
                        Subtitle area coordinates (ymin ymax xmin xmax). Can
                        be specified multiple times for multiple areas.
  --inpaint-mode {sttn-auto,sttn-det,lama,propainter,opencv}
                        Inpaint mode, default is sttn-auto


In [ ]:
# @title 直接运行
%cd {path}/video-subtitle-remover
!/content/.venv/bin/python ./backend/main.py -i test/input.mp4 -o test/output.mp4 --inpaint-mode sttn-det

In [ ]:
# @title 补丁:GPU字幕检测
%cd {path}/video-subtitle-remover
!/content/.venv/bin/python -c "from paddleocr import TextDetection; _init = TextDetection.__init__; TextDetection.__init__ = lambda self, *a, **k: _init(self, *a, **{**k, 'device': 'gpu' }); import sys; sys.argv = ['./backend/main.py', '-i', 'test/input.mp4', '-o', 'test/output.mp4', '--inpaint-mode', 'sttn-det']; import runpy; runpy.run_path('./backend/main.py', run_name='__main__')"

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.

📢 Tips: QFluentWidgets Pro is now released. Click https://qfluentwidgets.com/pages/pro to learn more about it.

ONNX provider: TensorrtExecutionProvider not supported, skipped.
Detected ONNX provider: CUDAExecutionProvider
Processing full screen (no area selected). Quality may vary.
Subtitle Area: [(0, 1080, 0, 1920)]
Processing block: All
Subtitle Removing:   0% 0/5529 [00:00<?, ?frame/s]Subtitle removal model: STTN Detection (GPU)
Subtitle detection model: Precise (CUDAExecutionProvider)
Subtitle Finding:   0% 0/5529 [00:00<?, ?frame/s][Processing] Detecting subtitles...
The Paddle Inference backend is selected with the default configuration. This may not provide optimal performance.
Using Paddle Inference backend
Paddle predictor option: device_type: gpu,  device_id: None,  run_mode: paddle,  trt_dynamic_shapes: {'x': [[1, 3, 32, 32], [1, 3